In [1]:
"""
精简索引检索
只需基本搜索功能      其余大半功能集成在watcher
       
index_manager.py - 知识库索引管理器
基准：fileUp.py (Engine) 输出结构
注意：Engine 输出的是扁平的卡片列表，字段名为 raw_header, category, subject, sequence 等。
"""

import json
import os
from typing import List, Dict, Optional
#04-1 全文搜索

def search_cards(query: str) -> List[Dict]:
    """
    全文搜索卡片（适配 Engine 输出结构）
    
    Args:
        query: 搜索关键词
        
    Returns:
        List[Dict]: 搜索结果列表，包含匹配分数
    """
    try:
        if not os.path.exists(DATA_FILE_PATH):
            print(f"警告: 数据文件 {DATA_FILE_PATH} 不存在，请先运行 Engine 生成数据。")
            return []
            
        with open(DATA_FILE_PATH, 'r', encoding='utf-8') as f:
            # Engine 输出的是直接的列表
            cards = json.load(f) 
    except Exception as e:
        print(f"读取卡片库失败: {e}")
        return []

    results = []
    query_lower = query.lower()
    
    
    for card in cards:
        score = 0
        
        # --- 字段适配：根据 Engine 的 JSON 结构修改 Key ---
        
        # 1. 标题匹配 (Engine 字段: raw_header)
        raw_title = card.get('raw_header', '').lower()
        if query_lower in raw_title:
            score += 3
            
        # 2. 编码匹配 (Engine 字段: category-subject-sequence)
        # 构建标准编码进行匹配
        cat = card.get('category', '').lower()
        subj = card.get('subject', '').lower()
        seq = card.get('sequence', '').lower()
        full_code = f"{cat}-{subj}-{seq}".lower()
        
        if query_lower in full_code:
            score += 2
            
        # 3. 内容匹配 (Engine 目前主要输出标题信息，若需内容匹配需配合文件读取)
        # 这里假设 Engine 后续会扩充 body 字段，或者我们简单用 raw_header 代替
        content_body = card.get('raw_header', '').lower() # 临时回退到标题
        # 如果 Engine 未来包含 'content' 字段，可以改为: 
        # content_body = card.get('content', {}).get('body', '').lower()
        
        if query_lower in content_body:
            score += 1

        if score > 0:
            # 为了兼容旧逻辑，临时包装一层 meta-like 结构用于显示，或者直接使用 Engine 结构
            # 这里我们直接在原数据上增加 _score
            card['_score'] = score
            results.append(card)

    # 按分数排序
    results.sort(key=lambda x: x.get('_score', 0), reverse=True)
    return results[:10]  # 限制结果数量


In [2]:
 
#04-2 查找调取
def get_card_by_code(code: str) -> Dict:
    """
    按编码获取卡片（适配版）
    
    Args:
        code: 卡片编码 (格式应为 Category-Subject-Sequence)
        
    Returns:
        Dict: 卡片数据或错误信息
    """
    try:
        if not os.path.exists(DATA_FILE_PATH):
            return {"error": "卡片库文件不存在"}

        with open(DATA_FILE_PATH, 'r', encoding='utf-8') as f:
            cards = json.load(f)
            
        # 解析输入的 code (例如: Python-装饰器-001)
        parts = code.strip().split('-')
        if len(parts) < 3:
            return {"error": "编码格式无效，应为 分类-主题-序号"}
            
        target_cat, target_subj, target_seq = parts[0], parts[1], parts[2]

        for card in cards:
            # --- 字段适配：直接比对 Engine 的字段 ---
            if (card.get('category') == target_cat and 
                card.get('subject') == target_subj and 
                card.get('sequence') == target_seq):
                
                # 为了符合 AI 的预期，我们可以临时构建一个包含 'meta' 的结构，
                # 或者直接返回 Engine 的原始结构。这里推荐直接返回原始结构以保持简洁。
                return card
                
    except Exception as e:
        return {"error": f"查询过程中发生错误: {str(e)}"}

    return {"error": f"未找到编码为 {code} 的卡片"}


In [4]:
#04-3 大类模糊查询

def list_cards_by_category(category: str) -> List[Dict]:
    """
    按分类列出卡片（适配版）
    
    Args:
        category: 分类名称 (如 'Python', '记忆')
        
    Returns:
        List[Dict]: 该分类下的所有卡片
    """
    try:
        with open(DATA_FILE_PATH, 'r', encoding='utf-8') as f:
            cards = json.load(f)
    except:
        return []

    results = []
    for card in cards:
        # --- 字段适配：Engine 使用 'category' 字段 ---
        card_category = card.get('category', '')
        if card_category == category: # 精确匹配分类名称
            results.append(card)
    
    return results

In [5]:
def refresh_index() -> Dict:
    """
    刷新索引
    提示：真正的刷新需要调用 Engine (fileUp.py) 重新解析文件。
    这里仅作为占位符或执行简单的文件重载。
    """
    try:
        # 如果未来实现内存缓存，这里可以清空缓存
        # 当前基于文件读取，无需特殊逻辑，只需提示用户去运行 Engine
        if os.path.exists(DATA_FILE_PATH):
            return {
                "status": "success", 
                "message": "索引文件已存在，如需更新请先运行 Engine (fileUp.py) 进行重新解析。"
            }
        else:
            return {
                "status": "warning", 
                "message": "索引文件不存在，请运行 Engine 生成数据。"
            }
    except Exception as e:
        return {"status": "error", "error": str(e)}


In [9]:
#04-4 接入ai  与额外测试
# --- CLI 接口 (用于调试) ---

if __name__ == "__main__":
    import sys
    
    if len(sys.argv) > 1:
        command = sys.argv[1]
        
        if command == "search" and len(sys.argv) > 2:
            query = sys.argv[2]
            results = search_cards(query)
            print(f"找到 {len(results)} 个结果:")
            for i, card in enumerate(results):
                # --- 适配打印：使用 Engine 的字段 ---
                cat = card.get('category', '未知')
                subj = card.get('subject', '未知')
                seq = card.get('sequence', '000')
                title = card.get('raw_header', '无标题')
                print(f"{i+1}. 【{cat}-{subj}-{seq}】{title}")
        
        elif command == "get" and len(sys.argv) > 2:
            code = sys.argv[2]
            card = get_card_by_code(code)
            if 'error' in card:
                print(f"错误: {card['error']}")
            else:
# --- 适配打印 ---
                raw_title = card.get('raw_header', '无标题')
                content = card.get('content', '无详细内容')
                print(f"找到卡片: {raw_title}")
                print(f"详细信息: {str(content)[:200]}...")
        
        elif command == "refresh":
            result = refresh_index()
            print(f"状态: {result['status']}")
            print(f"消息: {result.get('message', '')}")
        
        else:
            print("可用命令:")
            print("  search <关键词>   搜索卡片")
            print("  get <编码>        获取指定卡片 (格式: 分类-主题-序号)")
            print("  refresh           刷新索引状态")
            
    else:
        print("请提供命令参数 (search, get, refresh)")
        
        
        
# def import_ai():


可用命令:
  search <关键词>   搜索卡片
  get <编码>        获取指定卡片 (格式: 分类-主题-序号)
  refresh           刷新索引状态
